<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/week10_chunking_overlap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -O jsquad_train.json "https://raw.githubusercontent.com/yahoojapan/JGLUE/main/datasets/jsquad-v1.3/train-v1.3.json"

--2026-10-09 06:59:45--  https://raw.githubusercontent.com/yahoojapan/JGLUE/main/datasets/jsquad-v1.3/train-v1.3.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21737422 (21M) [application/octet-stream]
Saving to: ‘jsquad_train.json’

jsquad_train.json   100%[===================>]  20.73M  --.-KB/s    in 0.1s    

2026-10-09 06:59:45 (190 MB/s) - ‘jsquad_train.json’ saved [21737422/21737422]



In [ ]:
import json
with open("jsquad_train.json", "r") as f:
    squad = json.load(f)


In [ ]:
print(squad.keys())
art = squad["data"][0]
print("title:", art["title"])
p = art["paragraphs"][0]
print("context:", p["context"])
print("qas:", p["qas"][0])

dict_keys(['data'])
title: 造語
context: 造語 [SEP] 造語（ぞうご）は、新たに語（単語）を造ることや、既存の語を組み合わせて新たな意味の語を造ること、また、そうして造られた語である。新たに造られた語については、新語または新造語とも呼ばれる。
qas: {'question': '新たに語（単語）を造ることや、既存の語を組み合わせて新たな意味の語を造ること', 'id': 'a1000888p0q0', 'answers': [{'text': '造語', 'answer_start': 0}], 'is_impossible': False}


In [ ]:
SEP = " [SEP] "

def build_doc(article):
    prefix = article["title"] + SEP
    bodies, qas = [], []
    offset = 0                                   # 目前這段在長文件裡的起點
    stats = {"in_title": 0, "dropped": 0}
    for p in article["paragraphs"]:
        ctx = p["context"]
        assert ctx.startswith(prefix), ctx[:50]
        body = ctx[len(prefix):]                 # 拿掉「標題 [SEP] 」
        for qa in p["qas"]:
            text = qa["answers"][0]["text"]
            start = qa["answers"][0]["answer_start"] - len(prefix)
            if start < 0:                        # 答案標在標題上
                stats["in_title"] += 1
                start = body.find(text)
                if start == -1:
                    stats["dropped"] += 1
                    continue
            qas.append({"question": qa["question"], "answer": text,
                        "start": offset + start, "end": offset + start + len(text)})
        bodies.append(body)
        offset += len(body) + 1                  # +1 是段落間的換行
    return "\n".join(bodies), qas, stats

In [ ]:
import json, statistics

SEP = " [SEP] "

def build_doc(article):
    prefix = article["title"] + SEP
    bodies, qas = [], []
    offset = 0
    stats = {
        "start_pos": 0,      # start > 0：答案在本文中間（正常情況）
        "start_zero": 0,     # start = 0：答案在本文第一個字
        "in_title": 0,       # start < 0：答案標在貼紙上
        "dropped": 0,        # 標在貼紙上、本文也找不到 → 丟掉
        "ans_is_title": 0,   # 答案文字 = 文章標題
        "impossible": 0,     # 沒有答案的題目
        "bad_prefix": 0,     # 開頭不是預期的貼紙（理論上應該是 0）
    }

    for p in article["paragraphs"]:
        ctx = p["context"]
        if not ctx.startswith(prefix):           # 不當掉，記下來就好
            stats["bad_prefix"] += 1
            continue
        body = ctx[len(prefix):]                 # 撕掉貼紙

        for qa in p["qas"]:
            if qa.get("is_impossible", False) or not qa["answers"]:
                stats["impossible"] += 1
                continue

            text = qa["answers"][0]["text"]
            start = qa["answers"][0]["answer_start"] - len(prefix)

            if text == article["title"]:
                stats["ans_is_title"] += 1

            # 三種情況分類
            if start > 0:
                stats["start_pos"] += 1
            elif start == 0:
                stats["start_zero"] += 1
            else:                                # 負數：答案在貼紙上
                stats["in_title"] += 1
                start = body.find(text)
                if start == -1:
                    stats["dropped"] += 1
                    continue

            qas.append({
                "question": qa["question"],
                "answer": text,
                "start": offset + start,         # 換算成長文件位置
                "end": offset + start + len(text),
            })

        bodies.append(body)
        offset += len(body) + 1                  # +1 是換行

    return "\n".join(bodies), qas, stats


# ===== 對全部文章跑一次 =====
total = None
lengths = []
n_kept = 0
failures = []
docs = []                                        # 存起來，Step 2 要用

raw_total = sum(len(p["qas"]) for art in squad["data"] for p in art["paragraphs"])

for art in squad["data"]:
    doc, qas, stats = build_doc(art)

    if total is None:
        total = {k: 0 for k in stats}
    for k, v in stats.items():
        total[k] += v

    lengths.append(len(doc))
    n_kept += len(qas)

    # 驗證：每題的位置都要剛好抓到答案
    for q in qas:
        got = doc[q["start"]:q["end"]]
        if got != q["answer"]:
            failures.append((art["title"], q["answer"], got))

    docs.append({"title": art["title"], "doc": doc, "qas": qas})


# ===== 印出體檢報告 =====
def pct(x):
    return f"{x} ({x / raw_total:.1%})"

print("文章數:", len(squad["data"]))
print("原始題數:", raw_total)
print("保留題數:", pct(n_kept))
print()
print("start > 0:", pct(total["start_pos"]))
print("start = 0:", pct(total["start_zero"]))
print("start < 0（在貼紙上）:", pct(total["in_title"]))
print("  其中丟掉:", pct(total["dropped"]))
print("答案 = 標題:", pct(total["ans_is_title"]))
print("沒有答案:", pct(total["impossible"]))
print("開頭不是貼紙的段落:", total["bad_prefix"])
print()
q1, med, q3 = statistics.quantiles(lengths, n=4)
print(f"長文件長度（字）: 最短 {min(lengths)}｜25% {q1:.0f}｜中位數 {med:.0f}｜75% {q3:.0f}｜最長 {max(lengths)}")
print()
print("驗證失敗:", len(failures))
for f in failures[:5]:                           # 有錯的話先看前 5 個
    print("  ", f)

文章數: 710
原始題數: 62697
保留題數: 59568 (95.0%)

start > 0: 49871 (79.5%)
start = 0: 5840 (9.3%)
start < 0（在貼紙上）: 6986 (11.1%)
  其中丟掉: 3129 (5.0%)
答案 = 標題: 6485 (10.3%)
沒有答案: 0 (0.0%)
開頭不是貼紙的段落: 0

長文件長度（字）: 最短 48｜25% 948｜中位數 2264｜75% 5048｜最長 59624

驗證失敗: 0


In [ ]:
with open("jsquad_docs.json", "w") as f:
    json.dump(docs, f, ensure_ascii=False)


In [ ]:
ans_lens = [len(q["answer"]) for d in docs for q in d["qas"]]
print("答案長度 中位數:", statistics.median(ans_lens), "平均:", round(statistics.mean(ans_lens), 1), "最長:", max(ans_lens))

for L in [200, 400, 800]:
    pred = statistics.mean((a - 1) / L for a in ans_lens)
    print(f"L={L}, 不重疊，預測切斷率: {pred:.2%}")

答案長度 中位數: 5.0 平均: 6.1 最長: 19
L=200, 不重疊，預測切斷率: 2.57%
L=400, 不重疊，預測切斷率: 1.28%
L=800, 不重疊，預測切斷率: 0.64%


In [ ]:
def fixed_chunks(doc, size, overlap=0):
    """回傳每塊的 (起點, 終點)。"""
    step = size - overlap
    spans, s = [], 0
    while s < len(doc):
        e = min(s + size, len(doc))
        spans.append((s, e))
        if e == len(doc):
            break
        s += step
    return spans

def cut_rate(docs, size, overlap):
    cut = total = 0
    for d in docs:
        spans = fixed_chunks(d["doc"], size, overlap)
        for q in d["qas"]:
            total += 1
            inside = any(s <= q["start"] and q["end"] <= e for s, e in spans)
            if not inside:
                cut += 1
    return cut / total

for L in [200, 400, 800]:
    for ov in [0, int(L * 0.15)]:
        print(f"L={L}, overlap={ov}: 實際切斷率 {cut_rate(docs, L, ov):.2%}")

L=200, overlap=0: 實際切斷率 2.56%
L=200, overlap=30: 實際切斷率 0.00%
L=400, overlap=0: 實際切斷率 1.24%
L=400, overlap=60: 實際切斷率 0.00%
L=800, overlap=0: 實際切斷率 0.59%
L=800, overlap=120: 實際切斷率 0.00%


In [ ]:
print(model.max_seq_length)

NameError: name 'model' is not defined